# FastConformer 115M - VIVOS + Common Voice VI

Notebook này là lớp điều phối cho pipeline trong repo main. Code đã tách sang `ASR_local/src/asr_local`:

- `model/fastconformer.py`: cấu hình run VIVOS và Common Voice.
- `deploy/kaggle.py`: build/push/poll/pull Kaggle.
- `analytics/compare.py`: đọc kết quả, report, artifact manifest.

Account Kaggle mặc định: `trnmnhtn` (Manh Tan Tran).

Mặc định `RUN_REMOTE = False` để chỉ in lệnh, tránh tiêu quota Kaggle ngoài ý muốn.

## 0. Bootstrap package local + repo main

In [ ]:
from pathlib import Path
import sys


def find_local_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src" / "asr_local").is_dir():
            return candidate
    raise RuntimeError("Không tìm thấy ASR_local/src/asr_local")


LOCAL_ROOT = find_local_root()
LOCAL_SRC = LOCAL_ROOT / "src"
if str(LOCAL_SRC) not in sys.path:
    sys.path.insert(0, str(LOCAL_SRC))

from asr_local.paths import bootstrap

LOCAL_ROOT, MAIN_ROOT = bootstrap(LOCAL_ROOT)
print("LOCAL_ROOT =", LOCAL_ROOT)
print("MAIN_ROOT  =", MAIN_ROOT)

## 1. Cấu hình

In [ ]:
from asr_local.model.fastconformer import CommonVoiceRunConfig, FastConformerRunConfig

RUN_REMOTE = False  # đổi thành True khi muốn chạy Kaggle thật

vivos = FastConformerRunConfig()
cv = CommonVoiceRunConfig()
resume_nemo = cv.resume_nemo(MAIN_ROOT, vivos.run_id)

print("Chạy remote =", RUN_REMOTE)
print("Tham số VIVOS =", vivos.script_args())
print("Tham số CV    =", cv.script_args())
print("File .nemo resume =", resume_nemo)

## 2. Build + push fine-tune VIVOS

In [ ]:
from asr_local.deploy.kaggle import build_code_dataset, push_vivos_finetune

build_code_dataset(vivos, MAIN_ROOT, RUN_REMOTE)
push_vivos_finetune(vivos, MAIN_ROOT, RUN_REMOTE)

## 3. Poll + pull artifact VIVOS

In [ ]:
from asr_local.deploy.kaggle import poll_kernel, pull_kernel

poll_kernel(vivos.account, vivos.kernel, MAIN_ROOT, RUN_REMOTE)
pull_kernel(vivos.account, vivos.kernel, MAIN_ROOT, RUN_REMOTE)

## 4. Xem kết quả VIVOS

In [ ]:
from asr_local.analytics.compare import load_results, result_table

vivos_results = load_results(MAIN_ROOT, vivos.run_id)
result_table(vivos_results)

## 5. Upload `.nemo` resume + push Common Voice

In [ ]:
from asr_local.deploy.kaggle import push_common_voice_resume, upload_resume_dataset

upload_resume_dataset(cv, resume_nemo, MAIN_ROOT, RUN_REMOTE)
build_code_dataset(cv, MAIN_ROOT, RUN_REMOTE)
push_common_voice_resume(cv, MAIN_ROOT, RUN_REMOTE)

## 6. Poll + pull artifact Common Voice

In [ ]:
poll_kernel(cv.account, cv.kernel, MAIN_ROOT, RUN_REMOTE)
pull_kernel(cv.account, cv.kernel, MAIN_ROOT, RUN_REMOTE)

## 7. So sánh kết quả

In [ ]:
cv_results = load_results(MAIN_ROOT, cv.run_id)
result_table(vivos_results, cv_results)

## 8. Report main repo + scoreboard

In [ ]:
from asr_local.analytics.compare import run_main_report_commands

run_main_report_commands(MAIN_ROOT, vivos.run_id, cv.run_id)

## 9. Manifest artifact

In [ ]:
from asr_local.analytics.compare import artifact_manifest
import pandas as pd

pd.concat(
    [artifact_manifest(MAIN_ROOT, vivos.run_id), artifact_manifest(MAIN_ROOT, cv.run_id)],
    ignore_index=True,
)